# ?? Entrenamiento del Modelo Qwen para el Agente Inteligente de la Biblioteca Central UABC
### Proyecto MTAO - FCITEC UABC | Autor: L.S.C. Ana Gabriela Molina Moreno / Desarrollador: Adrián

Este notebook realiza el **Fine-Tuning ligero (LoRA)** del modelo **Qwen** utilizando la librería optimizada **Unsloth** sobre una GPU gratuita de Google Colab (T4).

**Objetivo:** Especializar a la IA para que tome consultas en lenguaje natural de alumnos de la UABC y las traduzca a formato estructurado JSON compatible con la **API de KOHA**.

## Paso 1: Instalar dependencias optimizadas (Unsloth)

In [ ]:
# Instalar Unsloth y dependencias optimizadas
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install unsloth_zoo
!pip install --no-deps trl peft accelerate bitsandbytes datasets

# Configuracion de LoRA Hiper-Especializado (r=32, alpha=64 para adaptacion profunda)
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
dtype = None # Auto detect (Float16 para T4)
load_in_4bit = True # Cuantización en 4-bit para bajo consumo de VRAM

# Cargamos el modelo Qwen 2.5 Instruct
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-0.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

from trl import SFTTrainer
from transformers import TrainingArguments

# Entrenamiento Profundo de Dominio (~2,956 ejemplos, 3 epocas completas)
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4, # Batch efectivo = 16
        warmup_steps = 25,
        max_steps = 550, # ~3 epocas completas sobre 2,956 muestras (~45-50 min en T4)
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("Iniciando entrenamiento profundo (550 pasos / ~2,956 ejemplos)...")
trainer_stats = trainer.train()
print("¡Entrenamiento profundo completado exitosamente!")

In [ ]:
# Configuracion de LoRA Hiper-Especializado (r=32, alpha=64 para adaptacion profunda)
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## Paso 4: Cargar el Dataset de la Biblioteca Central UABC
> Sube cualquiera de las 4 partes a la carpeta de archivos a la izquierda de Colab:
> - `dataset_koha_parte1_ingenieria.jsonl` (Ingeniería y Tecnología)
> - `dataset_koha_parte2_ciencias_sociales_derecho.jsonl` (Derecho y Humanidades)
> - `dataset_koha_parte3_administracion_economia.jsonl` (Administración y Negocios)
> - `dataset_koha_parte4_salud_ciencias_basicas.jsonl` (Salud y Ciencias)

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

def formatting_prompts_func(examples):
    convs = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convs]
    return { "text" : texts }

# Elige cuál de los 4 archivos subiste (o déjalo en parte 1):
archivo_dataset = "dataset_koha_parte1_ingenieria.jsonl"

dataset = load_dataset("json", data_files=archivo_dataset, split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"¡Dataset cargado ({archivo_dataset})! Muestras totales: {len(dataset)}")


## Paso 5: Entrenar el Modelo (Solo toma ~5 a 10 minutos)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Entrenamiento Profundo de Dominio (~2,956 ejemplos, 3 epocas completas)
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4, # Batch efectivo = 16
        warmup_steps = 25,
        max_steps = 450, # ~3 epocas sobre 2,500 muestras (~10-12 min en GPU T4)
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("Iniciando entrenamiento profundo (450 pasos / 2,500 ejemplos)...")
trainer_stats = trainer.train()
print("¡Entrenamiento profundo completado exitosamente!")

## Paso 6: Probar el Modelo Entrenado en Vivo

In [ ]:
FastLanguageModel.for_inference(model)

pregunta_alumno = "Oye disculpa, ¿tendrás el libro de redes de Tanenbaum disponible para préstamo?"

messages = [
    {"role": "system", "content": "Eres el asistente inteligente de la Biblioteca Central Campus Mexicali (UABC). Tu función es analizar la consulta del usuario en lenguaje natural y extraer la intención y parámetros de búsqueda para la API de KOHA en formato JSON."},
    {"role": "user", "content": pregunta_alumno}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 128, use_cache = True)
respuesta = tokenizer.batch_decode(outputs)
print("=== RESPUESTA DE LA IA ESPECIALIZADA ===")
print(respuesta[0].split("<|im_start|>assistant")[-1].replace("<|im_end|>", "").strip())

## Paso 7: Guardar y Exportar el Modelo a Formato GGUF para la UABC
Esta celda exporta el modelo cuantizado en formato `.gguf` (pesará ~400 a 700 MB) listo para descargarlo y meterlo al servidor local de la UABC.

In [ ]:
# Exportar a GGUF (q4_k_m para máxima velocidad y bajo peso)
model.save_pretrained_gguf("qwen_koha_uabc_model", tokenizer, quantization_method = "q4_k_m")
print("¡Modelo GGUF generado en la carpeta 'qwen_koha_uabc_model'!")